In [1]:
from pathlib import Path
import pickle
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from neuralhydrology.evaluation.metrics import calculate_metrics

In [2]:
# ------------------- Paths -------------------
RUN_DIR = Path("./runs")

ensemble_metrics_dir=Path("./ensemble_metrics")
ensemble_metrics_dir.mkdir(exist_ok=True)

In [3]:
# run_pattern = "mswep_precipitation_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed*"
# run_pattern = "chirps_precipitation_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed*"
# run_pattern = "total_precipitation_sum_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed*"
run_pattern = "camels_precipitation_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed*"
# run_pattern = "mswep_precipitation_chirps_precipitation_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed*"
# run_pattern = "chirps_v3_precipitation_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed*"
# run_pattern = "mswep_precipitation_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed*"

matched_paths = sorted(RUN_DIR.glob(f"{run_pattern}/validation/model_epoch030/validation_results.p"))
print(f"Found {len(matched_paths)} runs: {[p.parts[-4] for p in matched_paths]}")

Found 8 runs: ['camels_precipitation_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed111_2604_155636', 'camels_precipitation_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed222_2604_172244', 'camels_precipitation_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed333_2604_184908', 'camels_precipitation_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed444_2604_201525', 'camels_precipitation_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed555_2704_115742', 'camels_precipitation_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed666_2704_132410', 'camels_precipitation_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed777_2704_145030', 'camels_precipitation_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed888_2704_161646']


In [4]:
# Load all runs
all_runs_data = []
for file_path in matched_paths:
    with open(file_path, "rb") as f:
        all_runs_data.append(pickle.load(f))

# Average the simulated flows across seeds, per basin
ensemble_data = {}

for basin_id in all_runs_data[0].keys():
    # Stack simulated flows from all seeds: shape (n_seeds, n_timesteps, time_step)
    sims = np.stack([
        run[basin_id]['1D']['xr']['streamflow_sim'].values #['QObs(mm/d)_sim'].values 
        for run in all_runs_data
    ], axis=0)
    
    mean_sim = np.mean(sims, axis=0)  # Average across seeds
    
    # Copy structure from first run, replace sim with ensemble mean
    xr_ensemble = all_runs_data[0][basin_id]['1D']['xr'].copy()
    xr_ensemble['streamflow_sim'].values[:] = mean_sim #['QObs(mm/d)_sim'].values[:] = mean_sim #
    
    ensemble_data[basin_id] = {'1D': {'xr': xr_ensemble}}

ensemble_data

{'camels_01411300': {'1D': {'xr': <xarray.Dataset> Size: 58kB
   Dimensions:         (date: 3652, time_step: 1)
   Coordinates:
     * date            (date) datetime64[ns] 29kB 1989-10-01 ... 1999-09-30
     * time_step       (time_step) int64 8B 0
   Data variables:
       streamflow_obs  (date, time_step) float32 15kB 1.75 1.99 2.85 ... 0.46 0.55
       streamflow_sim  (date, time_step) float32 15kB 1.578 2.26 ... 0.5951 0.7052}},
 'camels_01491000': {'1D': {'xr': <xarray.Dataset> Size: 58kB
   Dimensions:         (date: 3652, time_step: 1)
   Coordinates:
     * date            (date) datetime64[ns] 29kB 1989-10-01 ... 1999-09-30
     * time_step       (time_step) int64 8B 0
   Data variables:
       streamflow_obs  (date, time_step) float32 15kB 1.39 1.86 3.26 ... 0.81 1.03
       streamflow_sim  (date, time_step) float32 15kB 1.555 3.612 ... 0.9332 1.234}},
 'camels_01644000': {'1D': {'xr': <xarray.Dataset> Size: 58kB
   Dimensions:         (date: 3652, time_step: 1)
   Coordinat

In [5]:
# Now compute metrics on the ensemble mean
all_metric_names = [
    'NSE', 'MSE', 'RMSE', 'KGE', 'Alpha-NSE', 'Pearson-r',
    'Beta-KGE', 'Beta-NSE', 'FHV', 'FMS', 'FLV',
    'Peak-Timing', 'Missed-Peaks', 'Peak-MAPE'
]

all_metrics = {}
for basin_id, basin_data in ensemble_data.items():
    xr_ds = basin_data['1D']['xr'].isel(time_step=0)
    
    all_metrics[basin_id] = calculate_metrics(
        obs=xr_ds['streamflow_obs'], #['QObs_mm_d_obs'],
        sim=xr_ds['streamflow_sim'], #['QObs_mm_d_sim'],
        metrics=all_metric_names,
        resolution="1D",
        datetime_coord="date"
    )

df_metrics = pd.DataFrame(all_metrics).T
df_metrics.index.name = 'basin_id'

df_metrics

,NSE,MSE,RMSE,KGE,Alpha-NSE,Pearson-r,Beta-KGE,Beta-NSE,FHV,FMS,FLV,Peak-Timing,Missed-Peaks,Peak-MAPE
basin_id,,,,,,,,,,,,,,
camels_01411300,0.715006,0.485379,0.696691,0.550992,0.604006,0.926882,0.801379,-0.199635,-36.578365,-4.819048e+00,2.476236e+01,0.250000,0.433962,35.270256
camels_01491000,0.791351,0.917411,0.957816,0.865188,0.921995,0.890153,1.004826,0.002748,-11.064801,7.864851e+00,5.654197e+01,0.333333,0.313725,45.853977
camels_01644000,0.796450,0.781822,0.884207,0.782181,0.843037,0.895975,0.890518,-0.059318,-9.212296,-9.396938e+00,9.021907e+01,0.045455,0.188679,32.670971
camels_01664000,0.781514,0.986407,0.993180,0.753685,0.832148,0.889951,0.857219,-0.084608,-13.556236,-3.053703e+00,7.243089e+01,0.095238,0.254237,36.225899
camels_01667500,0.750037,1.864795,1.365575,0.697957,0.756794,0.876634,0.870147,-0.063942,-13.921229,-2.695472e-01,7.525079e+01,0.000000,0.263158,34.011406
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
camels_09484600,0.186967,0.002708,0.052042,-0.902438,0.619276,0.552396,2.809412,0.337198,-35.192398,5.338664e+07,-1.033834e+12,0.615385,0.488372,68.649933
camels_09497980,0.758263,0.107565,0.327971,0.764538,0.899730,0.872288,0.829478,-0.043250,-12.000043,-8.688717e+00,-2.620456e+02,0.111111,0.432432,53.148399
camels_09510200,0.727941,0.384285,0.619907,0.624590,0.644648,0.886966,0.956628,-0.006980,-31.708496,-9.304315e+01,-3.860593e+10,0.333333,0.441176,60.391975


In [6]:
save_name = run_pattern.split("_seed")[0]
df_metrics.to_csv(f"./ensemble_metrics/{save_name}.csv")

In [7]:
df_metrics.median()

NSE              0.727941
MSE              1.500796
RMSE             1.225070
KGE              0.686015
Alpha-NSE        0.799496
Pearson-r        0.878639
Beta-KGE         0.876893
Beta-NSE        -0.054814
FHV            -19.744133
FMS             -8.979903
FLV             31.656826
Peak-Timing      0.315789
Missed-Peaks     0.342105
Peak-MAPE       44.391434
dtype: float64